In [ ]:
# Tile: design one encoder-decoder architecture from scratch
# requirements:
# 1. it can transform english sentences to another language like spanish
import torch
import torch.nn as nn

# vocab & token mapping
src_vocab = {"<PAD>": 0, "Pizza": 1, "is": 2, "great": 3, "!": 4}
tar_vocab = {"<PAD>": 0, "<SOS>": 1, "La": 2, "pizza": 3, "es": 4, "genial": 5, "!": 6, "<EOS>": 7}
idx2tgt = {v: k for k, v in tar_vocab.items()}

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        seq_len = x.size(1)
        pos = torch.arange(0, seq_len, dtype=torch.long, device=x.device).unsqueeze(0)
        x = x + self.pe(pos)
        return x
    
# encoder
class encoder(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.pos_encoding = PositionalEncoding(d_model)
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

    def forward(self, input_data):
        embedded_data = self.embedding(input_data)
        embedded_data = self.pos_encoding(embedded_data)

        # self-attention
        q = self.q_proj(embedded_data)
        k = self.k_proj(embedded_data)
        v = self.v_proj(embedded_data)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (q.size(-1) ** 0.5)
        attn_w = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_w, v)

        return context

# decoder
class decoder(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)

        # self-attention
        self.self_q_proj = nn.Linear(d_model, d_model)
        self.self_k_proj = nn.Linear(d_model, d_model)
        self.self_v_proj = nn.Linear(d_model, d_model)

        # cross-attention
        self.q_proj = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, len(tar_vocab))

    def forward(self, tar_input, encoder_output):
        tgt_embedded = self.embedding(tar_input)

        # self-attention
        q_self = self.self_q_proj(tgt_embedded)
        k_self = self.self_k_proj(tgt_embedded)
        v_self = self.self_v_proj(tgt_embedded)

        scores_self = torch.matmul(q_self, k_self.transpose(-2, -1)) / (q_self.size(-1) ** 0.5)

        seq_len = tar_input.size(1)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=tar_input.device), diagonal=1).bool()
        scores_self = scores_self.masked_fill(mask, float("-inf"))

        attn_w_self = torch.softmax(scores_self, dim=-1)
        tgt_context = torch.matmul(attn_w_self, v_self)

        # cross attention
        q = self.q_proj(tgt_context)
        k = encoder_output
        v = encoder_output

        scores = torch.matmul(q, k.transpose(-2, -1)) / (q.size(-1) ** 0.5)
        attn_w = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn_w, v)

        logits = self.fc_out(context)
        return logits

class seq2seq(nn.Module):
    def __init__(self, src_vocab_size, tar_vocab_size, d_model):
        super().__init__()
        self.encoder = encoder(src_vocab_size, d_model)
        self.decoder = decoder(tar_vocab_size, d_model)

    def forward(self, src_input, tar_input):
        encoder_output = self.encoder(src_input)
        logits = self.decoder(tar_input, encoder_output)
        return logits

class loss_fn(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, logits, target):
        # logits: [batch_size, seq_len, vocab_size]
        # target: [batch_size, seq_len]
        logits = logits.view(-1, logits.size(-1))
        target = target.view(-1)

        prob = torch.softmax(logits, dim=-1)
        correct_prob = prob[range(prob.size(0)), target]
        loss = -torch.log(correct_prob + 1e-9)

        # mask the padding tokens
        mask = target != tar_vocab["<PAD>"]
        loss = loss[mask]
        return loss.mean()


if __name__ == "__main__":
    src_input = torch.tensor([[src_vocab["Pizza"], src_vocab["is"], src_vocab["great"], src_vocab["!"]]])
    tar_input = torch.tensor([[tar_vocab["<SOS>"], tar_vocab["La"], tar_vocab["pizza"], tar_vocab["es"], tar_vocab["genial"], tar_vocab["!"]]])

    target = torch.tensor([[tar_vocab["La"], tar_vocab["pizza"], tar_vocab["es"], tar_vocab["genial"], tar_vocab["!"], tar_vocab["<EOS>"]]])

    model = seq2seq(len(src_vocab), len(tar_vocab), d_model=16)
    criterion = loss_fn()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # train
    for epoch in range(1500):
        logits = model(src_input, tar_input)
        loss = criterion(logits, target)

        print(f"Epoch {epoch}, Loss: {loss.item()}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    predi = torch.argmax(model(src_input, tar_input), dim=-1).squeeze(0).tolist()
    res_str = [idx2tgt[idx] for idx in predi]

    print(res_str)